In [1]:
import os, subprocess

# Pull PATH from a login shell that sources ~/.bashrc
os.environ["PATH"] = subprocess.check_output(
    ["bash", "-lc", "echo $PATH"],
    text=True
).strip()

# Sanity check
print(subprocess.check_output(["bash", "-lc", "which deno && deno --version"], text=True))

!which python
!which yt-dlp || echo "yt-dlp not on PATH"
!which ffmpeg || echo "ffmpeg not on PATH"
!deno --version


/home/hkanagalinga/.deno/bin/deno
deno 2.6.6 (stable, release, x86_64-unknown-linux-gnu)
v8 14.2.231.17-rusty
typescript 5.9.2

~/miniconda3/envs/he_gpu/bin/python
~/miniconda3/envs/he_gpu/bin/yt-dlp
~/miniconda3/envs/he_gpu/bin/ffmpeg
deno 2.6.6 (stable, release, x86_64-unknown-linux-gnu)
v8 14.2.231.17-rusty
typescript 5.9.2


In [2]:
import yt_dlp
import pysrt
import cv2
import os
import json
import pandas as pd
from pathlib import Path
import glob

# =========================
# Configuration
# =========================
CSV_DIRECTORY = "/data/scratch/projects/punim2198/heetha_scratch/new_data_2"
OUTPUT_BASE = "/data/scratch/projects/punim2198/heetha_scratch/videos"
SUBTITLE_LANG = "en"
MID_OFFSET_S = 0.0
FFMPEG_PATH = os.path.expanduser('~/ffmpeg-7.0.2-amd64-static/ffmpeg')

# =========================
# Utilities
# =========================
def ensure_dirs(path):
    os.makedirs(path, exist_ok=True)
    return path

def time_to_name(t_seconds: float) -> str:
    ms = int(round(t_seconds * 1000))
    return f"{ms:09d}ms"

def download_video_and_subtitles(video_url, output_dir, subtitle_lang):
    video_path = os.path.join(output_dir, "video.mp4")
    subtitle_path = os.path.join(output_dir, f"video.{subtitle_lang}.srt")

    if os.path.exists(video_path) and os.path.exists(subtitle_path):
        print("✔ Video + subtitles exist. Skipping download.")
        return video_path, subtitle_path

    print(f"⬇ Downloading {video_url} ...")
    ydl_opts = {
        "outtmpl": os.path.join(output_dir, "video.%(ext)s"),
        "writeautomaticsub": True,
        "writesubtitles": True,
        "subtitleslangs": [subtitle_lang],
        "subtitlesformat": "srt",
        "skip_download": False,
        "format": "bv*[vcodec^=avc1][height=480]+ba/best[ext=mp4][vcodec^=avc1][height=480]/bv*[vcodec^=avc1][height=720]+ba/best[ext=mp4][vcodec^=avc1][height=720]/bv*[vcodec^=avc1][height=360]+ba/best[ext=mp4][vcodec^=avc1][height=360]/best",
        "merge_output_format": "mp4",
        "cookiefile": "/data/scratch/projects/punim2198/heetha_scratch/cookies.txt",
    }
    if FFMPEG_PATH and os.path.exists(FFMPEG_PATH):
        ydl_opts["ffmpeg_location"] = FFMPEG_PATH

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=True)
        merged_ext = info.get("ext", "mp4")
        video_file = os.path.join(output_dir, f"video.{merged_ext}")
        if merged_ext.lower() != "mp4":
            target_path = os.path.join(output_dir, "video.mp4")
            if os.path.exists(video_file):
                os.replace(video_file, target_path)
            video_file = target_path

    if not os.path.exists(video_file):
        raise FileNotFoundError(f"Expected video not found: {video_file}")

    candidate_srt = os.path.join(output_dir, f"video.{subtitle_lang}.srt")
    if os.path.exists(candidate_srt):
        subtitle_file = candidate_srt
    else:
        all_srts = sorted(Path(output_dir).glob("video.*.srt"))
        if not all_srts:
            raise FileNotFoundError("No subtitles found.")
        subtitle_file = str(all_srts[0])

    return video_file, subtitle_file

def parse_subtitles(subtitle_file):
    subs = pysrt.open(subtitle_file, encoding="utf-8")
    def to_seconds(t): return t.ordinal / 1000.0
    return [
        {
            "index": idx,
            "text": sub.text.replace("\n", " ").strip(),
            "start_time": to_seconds(sub.start),
            "end_time": to_seconds(sub.end),
        }
        for idx, sub in enumerate(subs)
    ]

class VideoReader:
    def __init__(self, video_file):
        self.cap = cv2.VideoCapture(video_file)
        if not self.cap.isOpened():
            raise ValueError(f"Could not open {video_file}")
        self.fps = self.cap.get(cv2.CAP_PROP_FPS) or 30.0
        self.frame_count = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        self.duration = self.frame_count / self.fps if self.frame_count > 0 else None

    def extract_frame_at(self, t_seconds, out_path):
        if self.duration is not None:
            eps = 0.5 / self.fps
            t_seconds = min(max(0.0, t_seconds), max(0.0, self.duration - eps))
        frame_number = max(0, int(round(t_seconds * self.fps)))
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = self.cap.read()
        if ret:
            cv2.imwrite(out_path, frame)
            return frame
        return None

    def release(self):
        if self.cap: 
            self.cap.release()

def match_subtitles_to_frames(video_file, subtitle_data, frame_output_dir):
    subtitle_frame_map = {}
    vr = VideoReader(video_file)
    try:
        for sub in subtitle_data:
            mid_time = ((sub["start_time"] + sub["end_time"]) / 2.0) + MID_OFFSET_S
            fname = f"frame_{sub['index']:04d}_{time_to_name(mid_time)}.jpg"
            frame_path = os.path.join(frame_output_dir, fname)
            vr.extract_frame_at(mid_time, frame_path)
            subtitle_frame_map[sub["index"]] = {
                "text": sub["text"],
                "start_time": sub["start_time"],
                "end_time": sub["end_time"],
                "sample_time": mid_time,
                "frame_path": frame_path,
            }
    finally:
        vr.release()
    return subtitle_frame_map

# =========================
# Main loop for all CSVs
# =========================
def main():
    # Get all CSV files from the directory
    csv_files = glob.glob(os.path.join(CSV_DIRECTORY, "*.csv"))
    
    if not csv_files:
        print(f"❌ No CSV files found in {CSV_DIRECTORY}")
        return
    
    print(f"Found {len(csv_files)} CSV file(s):")
    for csv_file in csv_files:
        print(f"  - {os.path.basename(csv_file)}")
    print()
    
    total_videos = 0
    successful_videos = 0
    skipped_videos = 0
    failed_videos = 0
    
    for csv_file in csv_files:
        csv_name = os.path.splitext(os.path.basename(csv_file))[0]
        print(f"\n{'='*60}")
        print(f"Processing CSV: {csv_name}")
        print(f"{'='*60}")
        
        # Create output folder for this CSV
        csv_output_dir = ensure_dirs(os.path.join(OUTPUT_BASE, csv_name))
        
        try:
            df = pd.read_csv(csv_file)
            
            # Check if required columns exist
            if 'url' not in df.columns:
                print(f"⚠ Skipping {csv_name}: 'url' column not found")
                continue
            
            for idx, row in df.iterrows():
                total_videos += 1
                
                # Get video info
                vid_id = row.get("id", f"video_{idx}")
                title = row.get("title", "")
                channel = row.get("channel", "")
                url = row.get("url", "")
                
                if not url or pd.isna(url):
                    print(f"⚠ Row {idx}: Skipping empty URL")
                    failed_videos += 1
                    continue
                
                print(f"\n=== [{total_videos}] Processing {vid_id} | {title} ===")
                
                # Create folder structure for this video
                out_dir = ensure_dirs(os.path.join(csv_output_dir, str(vid_id)))
                frame_dir = ensure_dirs(os.path.join(out_dir, "frames"))
                map_path = os.path.join(out_dir, "subtitle_frame_map_middle.json")
                
                # Check if already fully processed
                if os.path.exists(map_path):
                    print(f"✔ Already processed (JSON exists). Skipping.")
                    skipped_videos += 1
                    successful_videos += 1
                    continue
                
                try:
                    # Download video and subtitles
                    video_file, subtitle_file = download_video_and_subtitles(url, out_dir, SUBTITLE_LANG)
                    
                    # Parse subtitles
                    subtitle_data = parse_subtitles(subtitle_file)
                    
                    # Extract frames
                    mapping = match_subtitles_to_frames(video_file, subtitle_data, frame_dir)
                    
                    # Keep only mappings where a non-empty frame file exists
                    non_empty_mapping = {
                        k: v for k, v in mapping.items()
                        if os.path.exists(v["frame_path"]) and os.path.getsize(v["frame_path"]) > 0
                    }
                    
                    # Save mapping to JSON
                    with open(map_path, "w", encoding="utf-8") as f:
                        json.dump(non_empty_mapping, f, indent=2, ensure_ascii=False)
                    
                    print(f"✅ Saved subtitle-frame map → {map_path}")
                    print(f"📊 Non-empty frames extracted: {len(non_empty_mapping)} / {len(subtitle_data)}")
                    
                    successful_videos += 1
                    
                except Exception as e:
                    print(f"❌ [ERROR] {vid_id}: {e}")
                    failed_videos += 1
        
        except Exception as e:
            print(f"❌ Error processing {csv_name}: {str(e)}")
            continue
    
    # Final summary
    print(f"\n{'='*60}")
    print("PROCESSING SUMMARY")
    print(f"{'='*60}")
    print(f"Total videos: {total_videos}")
    print(f"Successfully processed: {successful_videos}")
    print(f"Already completed (skipped): {skipped_videos}")
    print(f"Failed: {failed_videos}")
    print(f"Output directory: {OUTPUT_BASE}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()


Found 2 CSV file(s):
  - biology_reasoning_dataset.csv
  - physics_videos_from_cache.csv


Processing CSV: biology_reasoning_dataset

=== [1] Processing _dRXA1_e30o | 4. Molecular Genetics I ===
✔ Already processed (JSON exists). Skipping.

=== [2] Processing L4tEwAsVW0I | 25. Cancer 1 ===
✔ Already processed (JSON exists). Skipping.

=== [3] Processing SA8dRTq3qUA | 11. Cells, the Simplest Functional Units ===
✔ Already processed (JSON exists). Skipping.

=== [4] Processing jwZpVpV71Fg | Cancer Metabolism: From molecules to medicine ===
✔ Already processed (JSON exists). Skipping.

=== [5] Processing e0WZx7lUOrY | 6. Behavioral Genetics I ===
✔ Already processed (JSON exists). Skipping.

=== [6] Processing rZjwF5z-Xfw | 12. Genetics 1 – Cell Division &amp; Segregating Genetic Material ===
✔ Already processed (JSON exists). Skipping.

=== [7] Processing 62FdhX-zS2Y | 23. Stem Cells ===
✔ Already processed (JSON exists). Skipping.

=== [8] Processing FpXIGTFD8Qs | 30. Immunology 1 – Div

[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/7gLcuMtM_HY/video.en.srt
[download] 100% of   69.39KiB in 00:00:00 at 384.34KiB/s
[download] Resuming download at byte 10315377
[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/7gLcuMtM_HY/video.f135.mp4
[download] 100% of  157.78MiB in 00:00:56 at 2.81MiB/s     
[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/7gLcuMtM_HY/video.f251.webm
[download] 100% of   37.13MiB in 00:00:03 at 9.61MiB/s     
[Merger] Merging formats into "/data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/7gLcuMtM_HY/video.mp4"
Deleting original file /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/7gLcuMtM_HY/video.f251.webm (pass -k to keep)
Deleting original file /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_d

[info] _cduety3qVE: Downloading 1 format(s): 94
Deleting existing file /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/_cduety3qVE/video.en.vtt
[info] Writing video subtitles to: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/_cduety3qVE/video.en.vtt
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 4
[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/_cduety3qVE/video.en.vtt
[download] 100% of   30.13KiB in 00:00:01 at 28.41KiB/s                
[download] /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/_cduety3qVE/video.mp4 has already been downloaded
[download] 100% of   68.02MiB
❌ [ERROR] _cduety3qVE: No subtitles found.

=== [235] Processing ZhZvMbNT_qc | What is life? A Nobel Prize-winning scientist answers | Paul Nurse Full Interview ===
✔ Already processed (JSON exists). Skipping.

=== [236] Processing 2Jgb_Dp

[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/pXRQVyTjSTo/video.en.srt
[download] 100% of    4.37KiB in 00:00:00 at 26.94KiB/s
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 39
[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/pXRQVyTjSTo/video.mp4
[download] 100% of   33.70MiB in 00:00:17 at 1.95MiB/s                  
[FixupM3u8] Fixing MPEG-TS in MP4 container of "/data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/pXRQVyTjSTo/video.mp4"
✅ Saved subtitle-frame map → /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/pXRQVyTjSTo/subtitle_frame_map_middle.json
📊 Non-empty frames extracted: 41 / 41

=== [413] Processing OKjaMhv_k58 | Want to explore how microbiology impacts the world around us? | University of Southampton ===
✔ Already processed (JSON exists). Skipping.

=== [414] Processing t8FQ

[info] IVYKnOmnhCI: Downloading 1 format(s): 95
[info] Writing video subtitles to: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/IVYKnOmnhCI/video.en.vtt
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 5
[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/IVYKnOmnhCI/video.en.vtt


...

✅ Saved subtitle-frame map → /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/oZSaI8RHiuk/subtitle_frame_map_middle.json
📊 Non-empty frames extracted: 1554 / 1554

=== [2093] Processing 0wGs__d3J-c | Enzyme “atlas” helps researchers decipher cellular pathways ===
⬇ Downloading https://youtu.be/0wGs__d3J-c ...
[youtube] Extracting URL: https://youtu.be/0wGs__d3J-c
[youtube] 0wGs__d3J-c: Downloading webpage
[youtube] 0wGs__d3J-c: Downloading tv downgraded player API JSON
[youtube] 0wGs__d3J-c: Downloading web safari player API JSON
[youtube] 0wGs__d3J-c: Downloading player 652bbb94-main
[youtube] [jsc:deno] Solving JS challenges using deno
[youtube] 0wGs__d3J-c: Downloading m3u8 information
[info] 0wGs__d3J-c: Downloading subtitles: en
[info] 0wGs__d3J-c: Downloading 1 format(s): 136+251
[info] Writing video subtitles to: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/0wGs__d3J-c/video.en.srt


[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/0wGs__d3J-c/video.en.srt
[download] 100% of    2.13KiB in 00:00:00 at 8.76KiB/s
[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/0wGs__d3J-c/video.f136.mp4
[download] 100% of    3.87MiB in 00:00:01 at 3.08MiB/s   
[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/0wGs__d3J-c/video.f251.webm
[download] 100% of    1.39MiB in 00:00:00 at 1.58MiB/s   
[Merger] Merging formats into "/data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/0wGs__d3J-c/video.mp4"
Deleting original file /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/0wGs__d3J-c/video.f251.webm (pass -k to keep)
Deleting original file /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/0wGs__d3J-c/video.f136.mp4 (pass -k to keep)


[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/TkGDxZhk2aM/video.en.srt
[download] 100% of    1.54KiB in 00:00:00 at 6.42KiB/s
[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/TkGDxZhk2aM/video.f135.mp4
[download] 100% of    1.45MiB in 00:00:00 at 1.73MiB/s   
[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/TkGDxZhk2aM/video.f251.webm
[download] 100% of  749.17KiB in 00:00:00 at 1.02MiB/s   
[Merger] Merging formats into "/data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/TkGDxZhk2aM/video.mp4"
Deleting original file /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/TkGDxZhk2aM/video.f135.mp4 (pass -k to keep)
Deleting original file /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/TkGDxZhk2aM/video.f251.webm (pass -k to keep)


[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/X8cYr8WQUzw/video.en.srt
[download] 100% of    6.28KiB in 00:00:00 at 28.09KiB/s
[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/X8cYr8WQUzw/video.f136.mp4
[download] 100% of   13.98MiB in 00:00:04 at 3.05MiB/s     
[download] Destination: /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/X8cYr8WQUzw/video.f251.webm
[download] 100% of    6.90MiB in 00:00:00 at 7.62MiB/s   
[Merger] Merging formats into "/data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/X8cYr8WQUzw/video.mp4"
Deleting original file /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/X8cYr8WQUzw/video.f136.mp4 (pass -k to keep)
Deleting original file /data/scratch/projects/punim2198/heetha_scratch/videos/biology_reasoning_dataset/X8cYr8WQUzw/video.f251.webm (pass -k to kee

KeyboardInterrupt: 